In [2]:
import pandas as pd
import yfinance as yf
import numpy as np
from datetime import date, timedelta

In [9]:
def calculate_portfolio_volatility(weights, cov_matrix):
    cov_times = np.dot(weights, cov_matrix)
    variance = np.dot(weights, cov_times)
    volatility = np.sqrt(variance)
    return volatility

In [10]:
def min_variance_portfolio(mean_returns, cov_matrix):
    n_assets = len(mean_returns)

    # objective function: minimize portfolio volatility
    def objective(weights):
        return calculate_portfolio_volatility(weights, cov_matrix)
    
    # weights must sum to 1
    constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}

    # each weight between 0 and 1
    bounds = tuple((0,1) for _ in range(n_assets))

    # inital guess
    x0 = np.array([1/n_assets] * n_assets)

    from scipy.optimize import minimize
    result = minimize(objective, x0, method='SLSQP', bounds=bounds, constraints=constraints)

    optimal_weights = result.x
    return optimal_weights

In [12]:
def max_sharpe_portfolio(mean_returns, cov_matrix, risk_free_rate):
    n_assets = len(mean_returns)

    # maximizes sharpe ratio
    def objective(weights):
        portfolio_return = np.dot(weights, mean_returns)
        cov_times = np.dot(weights, cov_matrix)
        variance = np.dot(weights, cov_times)
        volatility = np.sqrt(variance)
        sharpe_ratio = (portfolio_return - risk_free_rate) / volatility
        return -(sharpe_ratio)

    constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
    bounds = tuple((0, 1) for _ in range(n_assets))
    x0 = np.array([1/n_assets] * n_assets)
    
    from scipy.optimize import minimize
    result = minimize(objective, x0, method='SLSQP', bounds=bounds, constraints=constraints)
    
    optimal_weights = result.x
    return optimal_weights

In [14]:
def risk_parity_portfolio(mean_returns, cov_matrix):
    n_assets = len(mean_returns)

    def objective(weights):
        cov_times = np.dot(weights, cov_matrix)
        port_vol = np.sqrt(np.dot(weights, cov_times))

        risk_contribution = weights * cov_times/ port_vol

        target_rc = np.sum(risk_contribution) / n_assets

        return np.sum((risk_contribution - target_rc)) ** 2

    constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
    bounds = tuple((0, 1) for _ in range(n_assets))
    x0 = np.array([1/n_assets] * n_assets)
    
    from scipy.optimize import minimize
    result = minimize(objective, x0, method='SLSQP', bounds=bounds, constraints=constraints)
    
    return result.x

In [17]:
def backtest_strategy(weights, prices, returns, rebalance_freq='Q'):
    """
    Backtest a strategy with periodic rebalancing
    
    Args:
        weights: initial portfolio weights
        prices: DataFrame of prices (from yfinance)
        returns: DataFrame of daily returns
        rebalance_freq: 'Q' (quarterly), 'M' (monthly), 'A' (annual)
    
    Returns:
        DataFrame with portfolio values over time
    """

    # tracks portfolio value
    portfolio_values = [1.0] # starting with $1
    dates = [prices.index[0]]
    current_weights = weights.copy()

    # determine rebalance period (trading days)
    if rebalance_freq == 'Q':
        rebalance_period = 63 #quarterly
    elif rebalance_freq == 'M':
        rebalance_period = 21 #monthly
    else:
        rebalance_period = 252 #annual

    # loop through each day
    for i in range(1, min(len(prices), len(returns))):
        day_returns = returns.iloc[i]

        # update portfolio value
        # portfolio value grows by weighted avg return
        pct_change = np.sum(current_weights * day_returns)
        new_value = portfolio_values[-1] * (1 + pct_change)
        portfolio_values.append(new_value)
        dates.append(prices.index[i])

        # rebalance on sched
        if i % rebalance_period == 0:
            # reset to initial weights
            current_weights = weights.copy()

    # Return as DataFrame
    backtest_df = pd.DataFrame({
        'date': dates,
        'portfolio_value': portfolio_values
    }).set_index('date')
    
    return backtest_df

In [18]:
def calculate_backtest_metrics(backtest_df, risk_free_rate=0.03):
    returns = backtest_df['portfolio_value'].pct_change().dropna()

    # CAGR (compound annual growth rate)
    total_return = backtest_df['portfolio_value'].iloc[-1] / backtest_df['portfolio_value'].iloc[0] - 1
    years = len(backtest_df) / 252
    cagr = (total_return + 1) ** (1/years) - 1

    volatility = returns.std() * np.sqrt(252)
    sharpe = (cagr - risk_free_rate) / volatility

    # maximize drawdown
    cummax = backtest_df['portfolio_value'].cummax()
    drawdown = (backtest_df['portfolio_value'] - cummax) / cummax
    max_dd = drawdown.min()

    return {
        'CAGR': cagr,
        'Volatility': volatility,
        'Sharpe': sharpe,
        'Max Drawdown': max_dd,
        'Total Return': total_return
    }

In [26]:
# ============================================================================
# VISUALIZATION FUNCTIONS
# ============================================================================

def plot_optimization_results(backtest_data, metrics_data):
    """
    Create 4-panel portfolio optimization comparison chart.
    
    Args:
        backtest_data: Dict of {strategy_name: backtest_df}
        metrics_data: Dict of {strategy_name: metrics_dict}
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Portfolio Optimizer: Strategy Comparison (5-Year Backtest)', 
                 fontsize=16, fontweight='bold')
    
    # Plot 1: Cumulative Returns
    ax = axes[0, 0]
    for name, df in backtest_data.items():
        ax.plot(df.index, df['portfolio_value'], label=name, linewidth=2)
    ax.set_title('Cumulative Returns (Starting Value = $1)')
    ax.set_ylabel('Portfolio Value ($)')
    ax.set_xlabel('Date')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot 2: Drawdown
    ax = axes[0, 1]
    for name, df in backtest_data.items():
        cummax = df['portfolio_value'].cummax()
        drawdown = (df['portfolio_value'] - cummax) / cummax * 100
        ax.plot(drawdown.index, drawdown, label=name, linewidth=2)
    ax.set_title('Drawdown Over Time')
    ax.set_ylabel('Drawdown (%)')
    ax.set_xlabel('Date')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot 3: Performance Metrics
    ax = axes[1, 0]
    names = list(metrics_data.keys())
    cagr_vals = [metrics_data[n]['CAGR'] * 100 for n in names]
    vol_vals = [metrics_data[n]['Volatility'] * 100 for n in names]
    sharpe_vals = [metrics_data[n]['Sharpe'] for n in names]
    
    x = np.arange(len(names))
    width = 0.25
    ax.bar(x - width, cagr_vals, width, label='CAGR (%)', alpha=0.8)
    ax.bar(x, vol_vals, width, label='Volatility (%)', alpha=0.8)
    ax.bar(x + width, sharpe_vals, width, label='Sharpe Ratio', alpha=0.8)
    ax.set_ylabel('Value')
    ax.set_title('Performance Metrics')
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=45, ha='right')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    # Plot 4: Risk vs Return Scatter
    ax = axes[1, 1]
    for name, metrics in metrics_data.items():
        ax.scatter(metrics['Volatility'] * 100, metrics['CAGR'] * 100, 
                  s=200, label=name, alpha=0.7)
        ax.annotate(name, (metrics['Volatility'] * 100, metrics['CAGR'] * 100),
                   xytext=(5, 5), textcoords='offset points', fontsize=9)
    ax.set_xlabel('Volatility (%)')
    ax.set_ylabel('Return (%)')
    ax.set_title('Risk vs Return')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('portfolio_optimization_results.png', dpi=300, bbox_inches='tight')
    print("✓ Optimization chart saved to 'portfolio_optimization_results.png'")
    plt.show()